In [1]:
import pandas as pd
import numpy as np
import csv
import json
import glob as glob_module
from pandas import json_normalize
from unidecode import unidecode
import os

COLS_ID = [
    "ano",
    "mes",
    "sigla_da_unidade",
]

COLS_BAIRROS_PAEFI = [
    "adalgisa_o", "alianca_o", "ayrosa_o", "bandeiras_o", "baronesa_o",
    "bela_vista_o", "bonanca_o", "bonfim_o", "bussocaba_o", "campesina_o",
    "castelo_branco_o", "centro_o", "cidade_das_flores_o", "cidade_de_deus_o",
    "cipava_o", "city_bussocaba_o", "conceicao_o", "helena_maria_o", "iapi_o",
    "industrial_altino_o", "industrial_anhanguera_o", "industrial_autono_o",
    "industrial_centro_o", "industrial_mazzei_o", "industrial_remedios_o",
    "jaguaribe_o", "jardim_d_abril_o", "jardim_elvira_o", "jardim_das_flores_o",
    "jardim_roberto_o", "km_18_o", "metalurgicos_o", "munhoz_junior_o",
    "mutinga_o", "novo_osasco_o", "outros_municipios_o", "padroeira_o",
    "paiva_ramos_o", "parque_continental_o", "pestana_o", "piratininga_o",
    "platina_o", "portal_d_oeste_o", "presidente_altino_o", "quitauna_o",
    "raposo_tavares_o", "remedios_o", "rochdale_o", "santa_fe_o", "santa_maria_o",
    "santo_antonio_o", "sao_pedro_o", "setor_militar_o", "tres_montanhas_o",
    "umuarama_o", "veloso_o", "vila_menck_o", "vila_militar_o", "vila_osasco_o",
    "vila_yara_o", "vila_yolanda_o",
]

COLS_BAIRROS_MSE = [
    "adalgisa_s", "alianca_s", "ayrosa_s", "bandeiras_s", "baronesa_s",
    "bela_vista_s", "bonanca_s", "bonfim_s", "bussocaba_s", "campesina_s",
    "castelo_branco_s", "centro_s", "cidade_das_flores_s", "cidade_de_deus_s",
    "cipava_s", "city_bussocaba_s", "conceicao_s", "helena_maria_s", "iapi_s",
    "industrial_altino_s", "industrial_anhanguera_s", "industrial_autonomistas_s",
    "industrial_centro_s", "industrial_mazzei_s", "industrial_remedios_s",
    "jaguaribe_s", "jardim_d_abril_s", "jardim_elvira_s", "jardim_das_flores_s",
    "jardim_roberto_s", "km_18_s", "metalurgicos_s", "munhoz_junior_s",
    "mutinga_s", "novo_osasco_s", "outros_municipios_s", "padroeira_s",
    "paiva_ramos_s", "parque_continental_s", "pestana_s", "piratininga_s",
    "platina_s", "portal_d_oeste_s", "presidente_altino_s", "quitauna_s",
    "raposo_tavares_s", "remedios_s", "rochdale_s", "santa_fe_s", "santa_maria_s",
    "santo_antonio_s", "sao_pedro_s", "setor_militar_s", "tres_montanhas_s",
    "umuarama_s", "veloso_s", "vila_menck_s", "vila_militar_s", "vila_osasco_s",
    "vila_yara_s", "vila_yolanda_s",
]

# Cor/raça do RF ou indivíduo inserido no PAEFI (seção N_1)
COLS_RACA_RF = [
    "branca_-_txt_cor_branca_masc_rf_individuo_n_1",
    "preta_-_txt_cor_preta_masc_rf_individuo_n_1",
    "parda_-_txt_cor_parda_masc_rf_individuo_n_1",
    "amarela_-_txt_cor_amarela_masc_rf_individuo_n_1",
    "indigena_-_txt_cor_indigena_masc_rf_individuo_n_1",
    "branca_-_txt_cor_branca_fem_rf_individuo_n_1",
    "preta_-_txt_cor_preta_fem_rf_individuo_n_1",
    "parda_-_txt_cor_parda_fem_rf_individuo_n_1",
    "amarela_-_txt_cor_amarela_fem_rf_individuo_n_1",
    "indigena_-_txt_cor_indigena_fem_rf_individuo_n_1",
]

# Cor/raça dos adolescentes em MSE (seção R_1)
COLS_RACA_MSE = [
    "branca_-_txt_cor_raca_adolesc_branca_masc_r_1",
    "preta_-_txt_cor_raca_adolesc_preta_masc_r_1",
    "parda_-_txt_cor_raca_adolesc_parda_masc_r_1",
    "amarela_-_txt_cor_raca_adolesc_amarela_masc_r_1",
    "indigena_-_txt_cor_raca_adolesc_indigena_masc_r_1",
    "branca_-_txt_cor_raca_adolesc_branca_fem_r_1",
    "preta_-_txt_cor_raca_adolesc_preta_fem_r_1",
    "parda_-_txt_cor_raca_adolesc_parda_fem_r_1",
    "amarela_-_txt_cor_raca_adolesc_amarela_fem_r_1",
    "indigena_-_txt_cor_raca_adolesc_indigena_fem_r_1",
]

# Identidade de gênero do RF/indivíduo (seção N_2)
COLS_ID_GENERO_RF = [
    "mulher_cis_-_txt_ident_genero_respons_familiar_mulher_cis_n_2",
    "homem_cis_-_txt_ident_genero_respons_familiar_homem_cis_n_2",
    "mulher_trans_-_txt_ident_genero_respons_familiar_mulher_trans_n_2",
    "homem_trans_-_txt_ident_genero_respons_familiar_homem_trans_n_2",
    "nao_binario_-txt_ident_genero_respons_familiar_nao_binario_n_2",
    "travesti_-_txt_ident_genero_respons_familiar_travesti_n_2",
    "intersexual_-_txt_ident_genero_respons_familiar_intersexual_n_2",
    "nao_declarado_-_txt_ident_genero_respons_familiar_nao_declarado_n_2",
]

# Identidade de gênero dos adolescentes em MSE (seção R_2)
COLS_ID_GENERO_MSE = [
    "mulher_cis_-_txt_identi_genero_adolesc_mulher_cis_r_2",
    "homem_cis_-_txt_identi_genero_adolesc_homem_cis_r_2",
    "mulher_trans_-_txt_identi_genero_adolesc_mulher_trans_r_2",
    "homem_trans_-_txt_identi_genero_adolesc_homem_trans_r_2",
    "nao_binario_-_txt_identi_genero_adolesc_nao_binario__r_2",
    "travesti_-_txt_identi_genero_adolesc_travesti_r_2",
    "intersexual_-_txt_identi_genero_adolesc_intersexual_r_2",
    "nao_declarado_-_txt_identi_genero_adolesc_nao_declarado_r_2",
]

DICT_ID_GENERO_RF = {
    "mulher_cis_-_txt_ident_genero_respons_familiar_mulher_cis_n_2": "Mulher Cis",
    "homem_cis_-_txt_ident_genero_respons_familiar_homem_cis_n_2": "Homem Cis",
    "mulher_trans_-_txt_ident_genero_respons_familiar_mulher_trans_n_2": "Mulher Trans",
    "homem_trans_-_txt_ident_genero_respons_familiar_homem_trans_n_2": "Homem Trans",
    "nao_binario_-txt_ident_genero_respons_familiar_nao_binario_n_2": "Não binário",
    "travesti_-_txt_ident_genero_respons_familiar_travesti_n_2": "Travesti",
    "intersexual_-_txt_ident_genero_respons_familiar_intersexual_n_2": "Intersexual",
    "nao_declarado_-_txt_ident_genero_respons_familiar_nao_declarado_n_2": "Não declarado",
}

DICT_ID_GENERO_MSE = {
    "mulher_cis_-_txt_identi_genero_adolesc_mulher_cis_r_2": "Mulher Cis",
    "homem_cis_-_txt_identi_genero_adolesc_homem_cis_r_2": "Homem Cis",
    "mulher_trans_-_txt_identi_genero_adolesc_mulher_trans_r_2": "Mulher Trans",
    "homem_trans_-_txt_identi_genero_adolesc_homem_trans_r_2": "Homem Trans",
    "nao_binario_-_txt_identi_genero_adolesc_nao_binario__r_2": "Não binário",
    "travesti_-_txt_identi_genero_adolesc_travesti_r_2": "Travesti",
    "intersexual_-_txt_identi_genero_adolesc_intersexual_r_2": "Intersexual",
    "nao_declarado_-_txt_identi_genero_adolesc_nao_declarado_r_2": "Não declarado",
}

MAP_SIGLA_UNIDADE = {
    "CREAS Norte": "CREAS Norte",
    "CREAS Sul": "CREAS Sul",
}

StatementMeta(, c49d6c1e-941a-40b2-8c0a-2144d820f3f4, 3, Finished, Available, Finished, False)

In [2]:
def normalizar_colunas(df):
    """Converte nomes de colunas para snake_case sem acentos."""
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[:()+,]", "", regex=True)
        .str.replace(r"[\s./'\']", "_", regex=True)
    )
    df.columns = [unidecode(col) for col in df.columns]
    return df


def safe_to_int(series, fill_value=0):
    """Converte série para int, tratando nulos e strings inválidas."""
    return pd.to_numeric(series, errors="coerce").fillna(fill_value).astype(int)

def ler_csv_rma(caminho="/lakehouse/default/Files/raw_sas_rma/bd_rma_creas.csv"):
    """Lê o CSV bruto de RMA-CREAS com tratamento de encoding e linhas inválidas."""
    return pd.read_csv(
        caminho,
        sep=";",
        encoding="utf-8-sig",
        engine="python",
        quotechar='"',
        doublequote=True,
        quoting=csv.QUOTE_MINIMAL,
        on_bad_lines="skip",
        dtype=str,
    )


def tratar_rmas(rma_acto, map_sigla_unidade):
    """Normaliza colunas e padroniza nomes de CREAS."""
    rma_acto = normalizar_colunas(rma_acto)
    rma_acto["mes"] = rma_acto["mes"].astype(int)

    # .replace() mantém valores não mapeados inalterados (ao contrário de .map())
    rma_acto["sigla_da_unidade"] = rma_acto["sigla_da_unidade"].replace(map_sigla_unidade)

    return rma_acto


RENAME_MAP_INDICADORES = {
    # ---------------------- SEÇÃO A — Acompanhamento PAEFI ----------------------
    "a_1_total_de_casos_familias_ou_individuos_em_acompanhamento_pelo_paefi":
        "A.1. Total de casos em acompanhamento pelo PAEFI",
    "a_2_novos_casos_familias_ou_individuos_inseridos_no_acompanhamento_do_paefi_durante_o_mes_de_referencia":
        "A.2. Novos casos inseridos no PAEFI durante o mês de referência",
    "a_3_total_de_casos_familias_ou_individuos_desligados_por_avaliacao_tecnica":
        "A.3. Total de casos desligados por avaliação técnica",
    "a_4_total_de_casos_familias_ou_individuos_desligados_por_evasao_ou_recusa_da_familia":
        "A.4. Total de casos desligados por evasão ou recusa da família",
    "a_5_total_de_casos_familias_ou_individuos_desligados_por_mudanca_de_regiao_ou_de_municipio":
        "A.5. Total de casos desligados por mudança de região ou de município",
    "a_6_total_de_casos_familias_ou_individuos_desligados_por_nao_localizacao":
        "A.6. Total de casos desligados por não localização",
    "a_7_total_de_casos_familias_ou_individuos_desligados_por_outros_motivos":
        "A.7. Total de casos desligados por outros motivos",
    "a_8_total_de_casos_familias_ou_individuos_desligados":
        "A.8. Total de casos desligados",
    # ---------------------- SEÇÃO B — Perfil dos casos em acompanhamento ----------------------
    "b_1_familias_beneficiarias_do_programa_bolsa_familia":
        "B.1. Famílias beneficiárias do Programa Bolsa Família",
    "b_2_familias_com_membros_beneficiarios_do_bpc":
        "B.2. Famílias com membros beneficiários do BPC",
    "b_3_familias_com_criancas_ou_adolescentes_em_situacao_de_trabalho_infantil":
        "B.3. Famílias com crianças ou adolescentes em situação de trabalho infantil",
    "b_3_1_total_de_familias_com_criancas_ou_adolescentes_em_situacao_de_trabalho_infantil_ate_15_anos_em_acompanhamento_no_creas":
        "B.3.1. Famílias com crianças ou adolescentes em trabalho infantil (até 15 anos) em acompanhamento no CREAS",
    "b_3_2_total_de_criancas_ou_adolescentes_em_situacao_de_trabalho_infantil_ate_15_anos_em_acompanhamento_no_creas":
        "B.3.2. Crianças ou adolescentes em trabalho infantil (até 15 anos) em acompanhamento no CREAS",
    "b_4_familias_com_criancas_ou_adolescentes_em_servicos_de_acolhimento_mesmo_ja_desacolhidas":
        "B.4. Famílias com crianças ou adolescentes em Serviços de Acolhimento (mesmo já desacolhidas)",
    "b_5_familias_cuja_situacao_de_violencia__violacao_esteja_associada_ao_uso_abusivo_de_substancias_psicoativas":
        "B.5. Famílias cuja situação de violência/violação esteja associada ao uso abusivo de substâncias psicoativas",
    "b_7_familias_com_adolescente_em_cumprimento_de_medidas_socioeducativas_em_meio_aberto":
        "B.7. Famílias com adolescente em cumprimento de Medidas Socioeducativas em meio aberto",
    "b_8_familias_beneficiarias_do_programa_nosso_futuro":
        "B.8. Famílias beneficiárias do Programa Nosso Futuro",
    "b_9_familias_imigrantes":
        "B.9. Famílias imigrantes",
    "b_10_familias_quilombolas":
        "B.10. Famílias quilombolas",
    "b_11_familias_ciganas":
        "B.11. Famílias ciganas",
    "b_12_familias_indigenas":
        "B.12. Famílias indígenas",
    "b_13_familias_de_outros_povos_ou_comunidades_tradicionais":
        "B.13. Famílias de outros povos ou comunidades tradicionais",
    "total_b_6_quantidade_de_pessoas_vitimadas_que_ingressaram_no_paefi_durante_o_mes_de_referencia_apenas_para_os_novos_casos":
        "B.6. Quantidade de pessoas vitimadas que ingressaram no PAEFI (novos casos)",
    # ---------------------- SEÇÃO C — Crianças e adolescentes (violência) ----------------------
    "total_-_c1_1_criancas_ou_adolescentes_vitimas_de_violencia_intrafamiliar_fisica":
        "C.1.1. Crianças ou adolescentes vítimas de violência intrafamiliar física",
    "total_-_txt_total_crianc_adolesc_vit_violenc_psicologica_c_1_2":
        "C.1.2. Crianças ou adolescentes vítimas de violência intrafamiliar psicológica",
    "total_-_c_1_=__c1_1___c_1_2_":
        "C.1. Crianças ou adolescentes vítimas de violência intrafamiliar (física ou psicológica)",
    "total_-_c_2_criancas_ou_adolescentes_vitimas_de_abuso_sexual":
        "C.2. Crianças ou adolescentes vítimas de abuso sexual",
    "total_-_c_3_criancas_ou_adolescentes_vitimas_de_exploracao_sexual":
        "C.3. Crianças ou adolescentes vítimas de exploração sexual",
    "total_-_c_4_criancas_ou_adolescentes_vitimas_de_negligencia_ou_abandono":
        "C.4. Crianças ou adolescentes vítimas de negligência ou abandono",
    "total_-_c_2_=_c_2__c_3__c_4_criancas_e_adolescentes_vitimas_de_abuso_sexual_exploracao_sexual_negligencia_ou_abandono":
        "C.2+C.3+C.4. Crianças e adolescentes vítimas de abuso sexual, exploração sexual, negligência ou abandono",
    "total_-_c_5_criancas_ou_adolescentes_em_situacao_de_trabalho_infantil_ate_15_anos":
        "C.5. Crianças ou adolescentes em situação de trabalho infantil (até 15 anos)",
    # ---------------------- SEÇÃO D — Pessoas idosas (violência) ----------------------
    "total_-_d_1_1_pessoas_idosas_vitimas_de_violencia_intrafamiliar_fisica":
        "D.1.1. Pessoas idosas vítimas de violência intrafamiliar física",
    "total_-_d_1_2_pessoas_idosas_vitimas_de_violencia_intrafamiliar_psicologica":
        "D.1.2. Pessoas idosas vítimas de violência intrafamiliar psicológica",
    "total_-_d_1_3_pessoas_idosas_vitimas_de_violencia_intrafamiliar_sexual":
        "D.1.3. Pessoas idosas vítimas de violência intrafamiliar sexual",
    "total_-_d_1=_d_1_1d_1_2d_1_3_fisica_psicologica_ou_sexual":
        "D.1. Pessoas idosas vítimas de violência intrafamiliar (física, psicológica ou sexual)",
    "total_-_d_2_pessoas_idosas_vitimas_de_negligencia_ou_abandono":
        "D.2. Pessoas idosas vítimas de negligência ou abandono",
    "total_-d_3_pessoas_idosas_vitimas_de_violencia_intrafamiliar_patrimonial":
        "D.3. Pessoas idosas vítimas de violência intrafamiliar patrimonial",
    "total_-_d_4_pessoas_idosas_vitimas_de_abuso_financeiro":
        "D.4. Pessoas idosas vítimas de abuso financeiro",
    # ---------------------- SEÇÃO E — Pessoas com deficiência (violência) ----------------------
    "total_-_e_1_1__pessoas_com_deficiencia_vitimas_de_violencia_intrafamiliar_fisica":
        "E.1.1. Pessoas com deficiência vítimas de violência intrafamiliar física",
    "total_-_e_1_2__pessoas_com_deficiencia_vitimas_de_violencia_intrafamiliar_psicologica":
        "E.1.2. Pessoas com deficiência vítimas de violência intrafamiliar psicológica",
    "total_-_e_1_3__pessoas_com_deficiencia_vitimas_de_violencia_intrafamiliar_sexual":
        "E.1.3. Pessoas com deficiência vítimas de violência intrafamiliar sexual",
    "total_-_txt_pessoas_defic_vit_violenc_intrafamiliar_fisic_psicolog_sexual_e_1":
        "E.1. Pessoas com deficiência vítimas de violência intrafamiliar (física, psicológica ou sexual)",
    "total_-_e_2__pessoas_com_deficiencia_vitimas_de_negligencia_ou_abandono":
        "E.2. Pessoas com deficiência vítimas de negligência ou abandono",
    "total_-_lbl_pessoas_defic_vit_viol_intrafamiliar_patrimonial_e_3":
        "E.3. Pessoas com deficiência vítimas de violência intrafamiliar patrimonial",
    "total_-_e_4__pessoas_com_deficiencia_vitimas_de_violencia_intrafamiliar_financeira":
        "E.4. Pessoas com deficiência vítimas de violência intrafamiliar financeira",
    "total_-_e_2_=_e_2__e_3__e_4":
        "E.2+E.3+E.4. Pessoas com deficiência vítimas de negligência, violência patrimonial ou financeira",
    # ---------------------- SEÇÃO F — Mulheres adultas ----------------------
    "f_1__mulheres_adultas_18_a_59_anos_vitimas_de_violencia_intrafamiliar_fisica_psicologica_ou_sexual":
        "F.1. Mulheres adultas (18 a 59 anos) vítimas de violência intrafamiliar (física, psicológica ou sexual)",
    # ---------------------- SEÇÃO G — Tráfico de seres humanos ----------------------
    "total_-_g_1__pessoas_vitimas_de_trafico_de_seres_humanos":
        "G.1. Pessoas vítimas de tráfico de seres humanos",
    # ---------------------- SEÇÃO H — Discriminação ----------------------
    "total_-_h__pessoas_vitimas_de_discriminacao_por_orientacao_sexual_que_ingressaram_no_paefi_durante_o_mes_de_referencia":
        "H. Pessoas vítimas de discriminação por orientação sexual que ingressaram no PAEFI",
    "h_1__pessoas_vitimas_de_discriminacao_por_orientacao_sexual":
        "H.1. Pessoas vítimas de discriminação por orientação sexual",
    "h_2__pessoas_vitimas_de_discriminacao_por_raca_etnia":
        "H.2. Pessoas vítimas de discriminação por raça/etnia",
    # ---------------------- SEÇÃO I — Situação de rua ----------------------
    "total_-_i_1__pessoas_em_situacao_de_rua":
        "I.1. Pessoas em situação de rua",
    # ---------------------- SEÇÃO M — Atendimentos ----------------------
    "m_1__total_de_atendimentos_individualizados_realizados_no_mes_de_referencia":
        "M.1. Total de atendimentos individualizados realizados no mês de referência",
    "m_1_1__total_de_atendimentos_para_triagem_e_ou_acolhimento":
        "M.1.1. Total de atendimentos para triagem e/ou acolhimento",
    "m_2__total_de_atendimentos_em_grupo_realizados_no_mes_de_referencia":
        "M.2. Total de atendimentos em grupo realizados no mês de referência",
    "m_3__familias_encaminhadas_para_o_cras_durante_o_mes_de_referencia":
        "M.3. Famílias encaminhadas para o CRAS durante o mês de referência",
    "m_4__visitas_domiciliares_realizadas_no_mes_de_referencia":
        "M.4. Visitas domiciliares realizadas no mês de referência",
    "m_5__concessao_de_beneficio_eventual_-_natalidade":
        "M.5. Concessão de benefício eventual — Natalidade",
    "m_6__concessao_de_beneficio_eventual_-_funeral":
        "M.6. Concessão de benefício eventual — Funeral",
    "m_7__concessao_de_beneficio_eventual_-_vulnerabilidade_temporaria":
        "M.7. Concessão de benefício eventual — Vulnerabilidade temporária",
    "m_8__concessao_de_beneficio_eventual_-_calamidade_publica":
        "M.8. Concessão de benefício eventual — Calamidade pública",
    # ---------------------- SEÇÃO J — Medidas Socioeducativas (MSE) ----------------------
    "j_1_total_de_adolescentes_em_cumprimento_de_medidas_socioeducativas_la_e_ou_psc":
        "J.1. Total de adolescentes em cumprimento de MSE (LA e/ou PSC)",
    "j_2_quantidade_de_adolescentes_em_cumprimento_de_liberdade_assistida_-_la":
        "J.2. Adolescentes em cumprimento de Liberdade Assistida (LA)",
    "j_3_quantidade_de_adolescentes_em_cumprimento_de_prestacao_de_servicos_a_comunidade_-_psc":
        "J.3. Adolescentes em cumprimento de Prestação de Serviços à Comunidade (PSC)",
    "total_-_j_4__total_de_novos_adolescentes_em_cumprimento_de_mse_la_e_ou_psc_inseridos_em_acompanhamento_no_mes_de_referencia":
        "J.4. Novos adolescentes em cumprimento de MSE (LA e/ou PSC) inseridos no mês de referência",
    "total_-_j_5__novos_adolescentes_em_cumprimento_de_la_inseridos_em_acompanhamento_no_mes_de_referencia":
        "J.5. Novos adolescentes em cumprimento de LA inseridos no mês de referência",
    "total_-_j_6__novos_adolescentes_em_cumprimento_de_psc_inseridos_em_acompanhamento_no_mes_de_referencia":
        "J.6. Novos adolescentes em cumprimento de PSC inseridos no mês de referência",
    "total_-_j_7__quantidade_de_adolescentes_primarios_em_cumprimento_de_mse_inseridos_em_acompanhamento_no_mes_de_referencia":
        "J.7. Adolescentes primários em cumprimento de MSE inseridos no mês de referência",
    "total_-_j_8__quantidade_de_adolescentes_reincidentes_em_cumprimento_de_mse_inseridos_em_acompanhamento_no_mes_de_referencia":
        "J.8. Adolescentes reincidentes em cumprimento de MSE inseridos no mês de referência",
    "total_-_j_9__quantidade_de_adolescentes_em_cumprimento_de_mse_em_situacao_de_trabalho_infantil":
        "J.9. Adolescentes em cumprimento de MSE em situação de trabalho infantil",
    "total_-_j_10__total_de_casos_desligados_por_realizacao_de_sua_finalidade":
        "J.10. Casos MSE desligados por realização de sua finalidade",
    "total_-_j_11__total_de_casos_desligados_por_aplicacao_de_pena_privativa_de_liberdade":
        "J.11. Casos MSE desligados por aplicação de pena privativa de liberdade",
    "total_-_j_12__total_de_casos_desligados_por_aplicacao_de_medida_socioeducativa_de_internacao":
        "J.12. Casos MSE desligados por aplicação de medida de internação",
    "total_-_j_13__total_de_casos_desligados_por_descumprimento_de_medida":
        "J.13. Casos MSE desligados por descumprimento de medida",
    "total_-_j_14__total_de_casos_desligados_por_condicao_de_doenca_grave":
        "J.14. Casos MSE desligados por condição de doença grave",
    "total_-_j_15__total_de_casos_desligados_por_morte_do_adolescente":
        "J.15. Casos MSE desligados por morte do adolescente",
    "total_-_j_16__total_de_casos_desligados_por_mudanca_de_regiao_ou_de_municipio":
        "J.16. Casos MSE desligados por mudança de região ou de município",
    "total_-_j_17__total_de_casos_desligados_por_nao_localizacao":
        "J.17. Casos MSE desligados por não localização",
    "total_-_j_18__total_de_atendimentos_realizados_no_mes":
        "J.18. Total de atendimentos MSE realizados no mês",
    "total_-_j_19__total_de_visitas_domiciliares_realizadas_no_mes":
        "J.19. Total de visitas domiciliares MSE realizadas no mês",
    # ---------------------- SEÇÃO K — Abordagem Social ----------------------
    "total_-_k_1__pessoas_abordadas_pelo_servico_de_abordagem_social_durante_o_mes_de_referencia":
        "K.1. Total de pessoas abordadas pelo Serviço de Abordagem Social",
    "k_2__criancas_ou_adolescentes_em_situacao_de_trabalho_infantil_ate_15_anos":
        "K.2. Crianças ou adolescentes em trabalho infantil (até 15 anos) abordados",
    "k_3__criancas_ou_adolescentes_em_situacao_de_exploracao_sexual":
        "K.3. Crianças ou adolescentes em situação de exploração sexual abordados",
    "k_4__criancas_ou_adolescentes_usuarios_de_crack_ou_outras_drogas":
        "K.4. Crianças ou adolescentes usuários de crack ou outras drogas abordados",
    "k_5__pessoas_adultas_usuarias_de_crack_ou_outras_drogas_ilicitas":
        "K.5. Pessoas adultas usuárias de crack ou outras drogas ilícitas abordadas",
    "k_6__migrantes":
        "K.6. Migrantes abordados",
    # ---------------------- SEÇÃO L — Abordagens ----------------------
    "l_1__quantidade_total_de_abordagens_realizadas_compreendida_como_o_numero_de_pessoas_abordadas_multiplicado_pelo_numero_de_vezes_em_que_foram_abordadas_durante_o_mes":
        "L.1. Quantidade total de abordagens realizadas no mês",
    # ---------------------- SEÇÃO P — Relatórios técnicos ----------------------
    "p_1__relatorios_tecnicos_elaborados_pela_equipe_do_paefi":
        "P.1. Relatórios técnicos elaborados pela equipe do PAEFI",
    "p_2__relatorios_tecnicos_elaborados_pela_equipe_de_mse-ma":
        "P.2. Relatórios técnicos elaborados pela equipe de MSE-MA",
    # ---------------------- SEÇÃO Q — Discussões de caso ----------------------
    "q_1__discussoes_de_caso_internas":
        "Q.1. Discussões de caso internas",
    "q_2__discussoes_de_caso_em_rede":
        "Q.2. Discussões de caso em rede",
}


def gerar_base_indicadores(rma_acto_tratado, cols_id, rename_map):
    """Gera base de indicadores no formato longo a partir das chaves do rename_map."""
    cols_indicadores = [c for c in rename_map if c in rma_acto_tratado.columns]

    indicadores = rma_acto_tratado[cols_id + cols_indicadores].melt(
        id_vars=cols_id,
        value_vars=cols_indicadores,
        var_name="indicador",
        value_name="valor",
    )
    indicadores["indicador"] = indicadores["indicador"].replace(rename_map)
    return indicadores


def _extrair_raca(nome_coluna):
    """Extrai a raça a partir do nome da coluna normalizada."""
    nome = nome_coluna.lower()
    if "parda" in nome:
        return "Parda"
    elif "preta" in nome:
        return "Preta"
    elif "amarela" in nome:
        return "Amarela"
    elif "branca" in nome:
        return "Branca"
    elif "indigena" in nome:
        return "Indígena"
    return None


def gerar_base_raca(rma_acto_tratado, cols_id, cols_raca):
    """Gera base de cor/raça no formato longo."""
    raca = (
        rma_acto_tratado[cols_id + cols_raca]
        .copy()
        .melt(
            id_vars=cols_id,
            value_vars=cols_raca,
            var_name="raca",
            value_name="valor",
        )
    )
    raca["genero"] = np.where(
        raca["raca"].str.contains("fem"), "Feminino", "Masculino"
    )
    raca["raca"] = raca["raca"].apply(_extrair_raca)
    return raca


def gerar_base_identidade_genero(rma_acto_tratado, cols_id, cols_id_genero, dict_id_genero):
    """Gera base de identidade de gênero no formato longo."""
    identidade_genero = (
        rma_acto_tratado[cols_id + cols_id_genero]
        .copy()
        .melt(
            id_vars=cols_id,
            value_vars=cols_id_genero,
            var_name="identidade_genero",
            value_name="valor",
        )
    )
    identidade_genero["identidade_genero"] = identidade_genero["identidade_genero"].replace(dict_id_genero)
    return identidade_genero


def gerar_base_bairros(rma_acto_tratado, cols_id, cols_bairros, strip_suffix=None):
    """Gera base de bairros de moradia no formato longo."""
    bairros = (
        rma_acto_tratado[cols_id + cols_bairros]
        .melt(
            id_vars=cols_id,
            value_vars=cols_bairros,
            var_name="bairro",
            value_name="valor",
        )
    )
    if strip_suffix:
        bairros["bairro"] = bairros["bairro"].str.removesuffix(strip_suffix)
    bairros["bairro"] = bairros["bairro"].str.replace("_", " ").str.title()
    return bairros

StatementMeta(, c49d6c1e-941a-40b2-8c0a-2144d820f3f4, 4, Finished, Available, Finished, False)

In [3]:
def main():
    rma_acto_origem = ler_csv_rma()
    rma_acto_tratado = tratar_rmas(rma_acto_origem, MAP_SIGLA_UNIDADE)

    # Validação: verificar se houve CREAS perdidos no mapeamento
    nulos_sigla = rma_acto_tratado["sigla_da_unidade"].isna().sum()
    if nulos_sigla > 0:
        print(f"AVISO: {nulos_sigla} registros com sigla_da_unidade nula após mapeamento.")
        valores_unicos = rma_acto_origem["sigla_da_unidade"].unique()
        mapeados = set(MAP_SIGLA_UNIDADE.keys())
        nao_mapeados = [v for v in valores_unicos if v not in mapeados and pd.notna(v)]
        if nao_mapeados:
            print(f"  Valores não mapeados: {nao_mapeados}")

    indicadores   = gerar_base_indicadores(rma_acto_tratado, COLS_ID, RENAME_MAP_INDICADORES)
    raca_rf       = gerar_base_raca(rma_acto_tratado, COLS_ID, COLS_RACA_RF)
    raca_mse      = gerar_base_raca(rma_acto_tratado, COLS_ID, COLS_RACA_MSE)
    id_genero_rf  = gerar_base_identidade_genero(rma_acto_tratado, COLS_ID, COLS_ID_GENERO_RF,  DICT_ID_GENERO_RF)
    id_genero_mse = gerar_base_identidade_genero(rma_acto_tratado, COLS_ID, COLS_ID_GENERO_MSE, DICT_ID_GENERO_MSE)
    bairros_paefi = gerar_base_bairros(rma_acto_tratado, COLS_ID, COLS_BAIRROS_PAEFI, strip_suffix="_o")
    bairros_mse   = gerar_base_bairros(rma_acto_tratado, COLS_ID, COLS_BAIRROS_MSE,   strip_suffix="_s")

    for df in [indicadores, raca_rf, raca_mse, id_genero_rf, id_genero_mse, bairros_paefi, bairros_mse]:
        df["valor"] = safe_to_int(df["valor"])

    print(f"Indicadores:        {indicadores.shape}")
    print(f"Raça RF:            {raca_rf.shape}")
    print(f"Raça MSE:           {raca_mse.shape}")
    print(f"Identidade gênero RF:  {id_genero_rf.shape}")
    print(f"Identidade gênero MSE: {id_genero_mse.shape}")
    print(f"Bairros PAEFI:      {bairros_paefi.shape}")
    print(f"Bairros MSE:        {bairros_mse.shape}")

    return (
        rma_acto_origem, rma_acto_tratado,
        indicadores,
        raca_rf, raca_mse,
        id_genero_rf, id_genero_mse,
        bairros_paefi, bairros_mse,
    )


(
    rma_acto_origem, rma_acto_tratado,
    indicadores,
    raca_rf, raca_mse,
    id_genero_rf, id_genero_mse,
    bairros_paefi, bairros_mse,
) = main()

StatementMeta(, c49d6c1e-941a-40b2-8c0a-2144d820f3f4, 5, Finished, Available, Finished, False)

Indicadores:        (1183, 5)
Raça RF:            (130, 6)
Raça MSE:           (130, 6)
Identidade gênero RF:  (104, 5)
Identidade gênero MSE: (104, 5)
Bairros PAEFI:      (793, 5)
Bairros MSE:        (793, 5)


# Histórico
- Ainda sem dados

In [4]:
def incluir_historico_raca(raca, cols_id, cols_raca, caminho_excel):
    """Inclui dados históricos de raça a partir de planilha Excel."""
    e2024 = pd.read_excel(caminho_excel)
    e2024 = normalizar_colunas(e2024)
    e2024 = e2024[cols_id + cols_raca].copy()
    e2024 = e2024.melt(
        id_vars=cols_id,
        value_vars=cols_raca,
        var_name="raca",
        value_name="valor",
    )
    e2024["genero"] = np.where(e2024["raca"].str.contains("fem"), "Feminino", "Masculino")
    e2024["raca"] = e2024["raca"].apply(_extrair_raca)
    e2024["valor"] = safe_to_int(e2024["valor"])
    return pd.concat([e2024, raca], ignore_index=True)

def incluir_historico_indicadores(indicadores, diretorio_parquets):
    """Inclui histórico de indicadores lendo todos os parquets do diretório via glob."""
    padrao = f"{diretorio_parquets}/rma_creas_indicadores_*.parquet"
    arquivos = glob_module.glob(padrao)
    if not arquivos:
        print(f"Nenhum parquet encontrado em: {padrao}")
        return indicadores
    historico = pd.concat(
        [pd.read_parquet(a) for a in arquivos],
        ignore_index=True,
    )
    return pd.concat([historico, indicadores], ignore_index=True)


# CONSOLIDACAO_INDICADORES: normaliza variações de rótulo ao longo do histórico.
# Preencher conforme variações forem identificadas nos dados históricos.
CONSOLIDACAO_INDICADORES = {
    # A.1
    "A.1. Total de casos em acompanhamento pelo PAEFI":
        "A.1. Total de casos em acompanhamento pelo PAEFI",
    # A.2
    "A.2. Novos casos inseridos no PAEFI durante o mês de referência":
        "A.2. Novos casos inseridos no PAEFI durante o mês de referência",
    # A.8
    "A.8. Total de casos desligados":
        "A.8. Total de casos desligados",
    # J.1
    "J.1. Total de adolescentes em cumprimento de MSE (LA e/ou PSC)":
        "J.1. Total de adolescentes em cumprimento de MSE (LA e/ou PSC)",
    # M.1
    "M.1. Total de atendimentos individualizados realizados no mês de referência":
        "M.1. Total de atendimentos individualizados realizados no mês de referência",
}

indicadores_total = indicadores.copy()
# indicadores_total = incluir_historico_indicadores(indicadores, "/lakehouse/default/Files/historico_creas")
indicadores_total["indicador"] = indicadores_total["indicador"].replace(CONSOLIDACAO_INDICADORES)

indicadores_total["indicador_media"] = indicadores_total["indicador"].replace(
    "A.1. Total de casos em acompanhamento pelo PAEFI",
    "A.1. Média de casos em acompanhamento pelo PAEFI",
)

print(f"Indicadores total: {indicadores_total.shape}")


indicadores_total = indicadores_total.astype({
    "ano": int,
    "mes": int,
    "sigla_da_unidade": str,
    "indicador": str,
    "indicador_media": str,
})

for df in [raca_rf, raca_mse]:
    df["ano"] = df["ano"].astype(int)
    df["mes"] = df["mes"].astype(int)
    df["sigla_da_unidade"] = df["sigla_da_unidade"].astype(str)
    df["raca"] = df["raca"].astype(str)
    df["genero"] = df["genero"].astype(str)

for df in [id_genero_rf, id_genero_mse]:
    df["ano"] = df["ano"].astype(int)
    df["mes"] = df["mes"].astype(int)
    df["sigla_da_unidade"] = df["sigla_da_unidade"].astype(str)
    df["identidade_genero"] = df["identidade_genero"].astype(str)

for df in [bairros_paefi, bairros_mse]:
    df["ano"] = df["ano"].astype(int)
    df["mes"] = df["mes"].astype(int)
    df["sigla_da_unidade"] = df["sigla_da_unidade"].astype(str)
    df["bairro"] = df["bairro"].astype(str)
    df["valor"] = df["valor"].astype(int)



StatementMeta(, c49d6c1e-941a-40b2-8c0a-2144d820f3f4, 6, Finished, Available, Finished, False)

Indicadores total: (1183, 6)


In [5]:
def validar_dados(rma_acto_tratado, indicadores, raca_rf, raca_mse,
                  id_genero_rf, id_genero_mse, bairros_paefi, bairros_mse):
    """Executa verificações de integridade nos dados processados."""
    erros = []

    # 1. Verificar NaN em sigla_da_unidade
    nulos = rma_acto_tratado["sigla_da_unidade"].isna().sum()
    if nulos > 0:
        erros.append(f"sigla_da_unidade com NaN: {nulos} registros")

    # 2. Verificar siglas não mapeadas
    siglas_validas = set(MAP_SIGLA_UNIDADE.values())
    siglas_encontradas = set(rma_acto_tratado["sigla_da_unidade"].dropna().unique())
    nao_mapeadas = siglas_encontradas - siglas_validas
    if nao_mapeadas:
        erros.append(f"Siglas fora do MAP_SIGLA_UNIDADE: {nao_mapeadas}")

    # 3. Verificar colunas do RENAME_MAP não encontradas no CSV
    ausentes = [c for c in RENAME_MAP_INDICADORES if c not in rma_acto_tratado.columns]
    if ausentes:
        erros.append(f"Colunas do RENAME_MAP ausentes no CSV ({len(ausentes)}): {ausentes[:5]}...")

    # 4. Verificar valores negativos nos indicadores
    negativos = (indicadores["valor"] < 0).sum()
    if negativos > 0:
        erros.append(f"Indicadores com valor negativo: {negativos}")

    # 5. Verificar raças nulas (coluna não reconhecida)
    for nome, df in [("raca_rf", raca_rf), ("raca_mse", raca_mse)]:
        nulas = df["raca"].isna().sum()
        if nulas > 0:
            erros.append(f"{nome}: {nulas} raças nulas (colunas não reconhecidas)")

    if erros:
        print("ERROS DE VALIDAÇÃO:")
        for e in erros:
            print(f"  - {e}")
    else:
        print("Validação concluída sem erros.")

    return erros


erros = validar_dados(
    rma_acto_tratado,
    indicadores,
    raca_rf, raca_mse,
    id_genero_rf, id_genero_mse,
    bairros_paefi, bairros_mse,
)

StatementMeta(, c49d6c1e-941a-40b2-8c0a-2144d820f3f4, 7, Finished, Available, Finished, False)

Validação concluída sem erros.


In [6]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
)

schema_indicadores = StructType([
    StructField("ano",              IntegerType(), True),
    StructField("mes",              IntegerType(), True),
    StructField("sigla_da_unidade", StringType(),  True),
    StructField("indicador",        StringType(),  True),
    StructField("valor",            DoubleType(),  True),
    StructField("indicador_media",  StringType(),  True),
])
spark.createDataFrame(indicadores_total, schema=schema_indicadores) \
    .write.mode("overwrite").format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_rma_creas_indicadores")


# schema_raca = StructType([
#     StructField("ano",              IntegerType(), True),
#     StructField("mes",              IntegerType(), True),
#     StructField("sigla_da_unidade", StringType(),  True),
#     StructField("raca",             StringType(),  True),
#     StructField("valor",            IntegerType(), True),
#     StructField("genero",           StringType(),  True),
# ])
# spark.createDataFrame(raca_rf, schema=schema_raca) \
#     .write.mode("overwrite").format("delta") \
#     .option("overwriteSchema", "true") \
#     .saveAsTable("gold_rma_creas_raca_rf")

# spark.createDataFrame(raca_mse, schema=schema_raca) \
#     .write.mode("overwrite").format("delta") \
#     .option("overwriteSchema", "true") \
#     .saveAsTable("gold_rma_creas_raca_mse")


# schema_id_genero = StructType([
#     StructField("ano",              IntegerType(), True),
#     StructField("mes",              IntegerType(), True),
#     StructField("sigla_da_unidade", StringType(),  True),
#     StructField("identidade_genero",StringType(),  True),
#     StructField("valor",            IntegerType(), True),
# ])
# spark.createDataFrame(id_genero_rf, schema=schema_id_genero) \
#     .write.mode("overwrite").format("delta") \
#     .option("overwriteSchema", "true") \
#     .saveAsTable("gold_rma_creas_id_genero_rf")

# spark.createDataFrame(id_genero_mse, schema=schema_id_genero) \
#     .write.mode("overwrite").format("delta") \
#     .option("overwriteSchema", "true") \
#     .saveAsTable("gold_rma_creas_id_genero_mse")
# schema_bairros = StructType([
#     StructField("ano",              IntegerType(), True),
#     StructField("mes",              IntegerType(), True),
#     StructField("sigla_da_unidade", StringType(),  True),
#     StructField("bairro",           StringType(),  True),
#     StructField("valor",            IntegerType(), True),
# ])
# spark.createDataFrame(bairros_paefi, schema=schema_bairros) \
#     .write.mode("overwrite").format("delta") \
#     .option("overwriteSchema", "true") \
#     .saveAsTable("gold_rma_creas_bairros_paefi")

# spark.createDataFrame(bairros_mse, schema=schema_bairros) \
#     .write.mode("overwrite").format("delta") \
#     .option("overwriteSchema", "true") \
#     .saveAsTable("gold_rma_creas_bairros_mse")

StatementMeta(, c49d6c1e-941a-40b2-8c0a-2144d820f3f4, 8, Finished, Available, Finished, False)